# MTMM — Cross-species Microsomal Stability Predictor (Colab)

A browser-based inference notebook. **No local installation required.**
It predicts liver microsomal metabolic stability for **human (HLM) / rat (RLM) / mouse (MLM)**
from SMILES and SwissADME descriptors.

- Label convention: `0 = stable`, `1 = unstable` (binarization cutoff t½ ≤ 30 min)
- Input requires a **SMILES column + SwissADME descriptor columns**.
  SwissADME (http://www.swissadme.ch/) is a web tool; some predicted columns
  (CYP/BBB/P-gp/GI) cannot be reproduced locally, so obtain the CSV from SwissADME first and merge it.

**Steps**: 1) setup → 2) get code & bundle → 3) upload input → 4) predict → 5) save results → 6) explain

> For speed, pick a GPU via *Runtime → Change runtime type* (CPU also works).

## 1. Environment setup (install)

In [ ]:
# Colab already has torch installed; only PyG and RDKit are added.
# Recent torch_geometric (>=2.3) runs GATv2Conv / GINEConv / pooling /
# JumpingKnowledge with pure-PyTorch fallbacks (no torch-scatter / torch-sparse).
!pip install -q torch_geometric rdkit shap

import torch, torch_geometric, rdkit
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("torch_geometric:", torch_geometric.__version__, "| rdkit:", rdkit.__version__)

# (Rarely) if PyG raises a scatter-related error, install the following for your torch:
# !pip install -q pyg-lib torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html

## 2. Get the model code and release bundle

Inference needs two things:
1. **Code**: `model.py`, `dataset_scaffold_modelready.py`, `predict.py`
2. **Release bundle** `mtmm_release.pt` (model + descriptor scaler/impute/schema + per-species thresholds),
   built once with `export_release_bundle.py` in the training notebook.

The repository and Release download URLs are already filled in. If you have not yet uploaded the
bundle to the Release (`v1.0-revision`), use the **upload option** (Option C) at the bottom of each cell.

In [ ]:
import sys, os, glob

# ── Option A: clone the code from GitHub (recommended) ────────────────
REPO_URL = "https://github.com/bmil-jnu/cross-species_metabolic_stability_prediction.git"   # edit if your repo path differs
CODE_DIR = "/content/repo"
if not os.path.exists(CODE_DIR):
    !git clone -q "$REPO_URL" "$CODE_DIR"

# Find the folder that actually contains model.py (works whether the code is at the
# repo root or inside a subfolder such as Main/ or Notebooks/), and add it to sys.path.
hits = glob.glob(os.path.join(CODE_DIR, "**", "model.py"), recursive=True)
assert hits, "model.py not found in the cloned repo (check REPO_URL / branch)."
# Prefer the folder that also has the inference + explanation modules.
def _score(d):
    return sum(os.path.exists(os.path.join(d, f))
               for f in ["predict.py", "explain.py", "dataset_scaffold_modelready.py"])
code_path = max((os.path.dirname(h) for h in hits), key=_score)
sys.path.insert(0, code_path)
print("using code from:", code_path)

# ── Option B (alternative): upload the code files if you have no repo ──
#   upload model.py / dataset_scaffold_modelready.py / predict.py / explain.py
# from google.colab import files
# files.upload()
# sys.path.insert(0, "/content")

# check imports
import model  # noqa: F401  (needed so the bundle unpickles model.MTMM)
from predict import load_bundle, run_inference
from dataset_scaffold_modelready import PHYS_COLS, CONT_COLS, CAT_COLS, find_smiles_column
print("code loaded")

In [ ]:
# Download the release bundle
BUNDLE_PATH = "/content/mtmm_release.pt"

# ── Option A: download from a GitHub Release asset ──
BUNDLE_URL = "https://github.com/bmil-jnu/cross-species_metabolic_stability_prediction/releases/download/v1.0-revision/mtmm_release.pt"  # edit if your release tag differs
if not os.path.exists(BUNDLE_PATH):
    !wget -q -O "$BUNDLE_PATH" "$BUNDLE_URL"

# ── Option B (alternative): download from Google Drive ──
# !pip install -q gdown
# import gdown; gdown.download(id="<GDRIVE_FILE_ID>", output=BUNDLE_PATH, quiet=True)  # TODO

# ── Option C (alternative): upload directly ──
# from google.colab import files
# up = files.upload(); BUNDLE_PATH = "/content/" + list(up.keys())[0]

assert os.path.exists(BUNDLE_PATH) and os.path.getsize(BUNDLE_PATH) > 0, \
    "Bundle not found. Fill in the URL above or use the upload option."
print("bundle:", BUNDLE_PATH, "|", os.path.getsize(BUNDLE_PATH), "bytes")

## 3. Upload the input CSV (SMILES + SwissADME)

- The SMILES column must be named one of `Cano_Smile`, `SMILES`, `PUBCHEM_EXT_DATASOURCE_SMILES`.
- Use the SwissADME descriptor columns exactly as exported from the SwissADME web tool.
- Missing SwissADME columns are imputed with training values, but the more are missing, the further from the validated setting.

In [ ]:
from google.colab import files
import pandas as pd

print("Upload a CSV containing SMILES + SwissADME columns.")
up = files.upload()
INPUT_CSV = "/content/" + list(up.keys())[0]
df = pd.read_csv(INPUT_CSV)
print("rows:", len(df), "| cols:", len(df.columns))

# check required columns
required = PHYS_COLS + CONT_COLS + CAT_COLS
missing = [c for c in required if c not in df.columns]
smi_col = find_smiles_column(df)

print("SMILES column:", smi_col)
assert smi_col is not None, "SMILES column not found (check allowed headers)."
if missing:
    print(f"⚠️ Missing SwissADME columns ({len(missing)}/{len(required)}):")
    print("   ", missing)
else:
    print("✅ All SwissADME columns present")

## 4. Run prediction

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bundle = load_bundle(BUNDLE_PATH, device)
print("tasks:", bundle["tasks"], "| thresholds:", bundle["thresholds"])

preds = run_inference(bundle, df, device, batch_size=64)
preds.head(20)

## 5. Save / download results

In [ ]:
OUT_CSV = "predictions.csv"
preds.to_csv(OUT_CSV, index=False)
print("saved:", OUT_CSV, "| rows:", len(preds))

from google.colab import files
files.download(OUT_CSV)

## 6. Model explanations (SHAP / EdgeSHAPer)

These cells reuse the `bundle`, `df`, and `device` loaded above to produce the two
interpretability outputs from the paper, with figures shown inline:

- **Method S5 — descriptor SHAP**: which ADME / physicochemical descriptors drive the prediction.
- **Method S6 — EdgeSHAPer**: per-bond structural contribution drawn on the molecule
  (blue = stabilizing, red = destabilizing).

> SHAP is compute-heavy; a GPU runtime is recommended. The demo settings below are modest for speed.

In [ ]:
from explain import descriptor_shap, plot_descriptor_shap, edgeshaper, draw_edge_attribution
from IPython.display import Image, display
import os

SPECIES = "human"            # one of: human / rat / mouse
EXPLAIN_DIR = "explain_out"
os.makedirs(EXPLAIN_DIR, exist_ok=True)
print("explaining species:", SPECIES)

In [ ]:
# Method S5 — descriptor-level KernelSHAP
# Demo settings are modest for speed; raise toward 64 / 256 / 2048 for manuscript-grade fidelity.
sv, ex, names, ev = descriptor_shap(
    bundle, df, SPECIES,
    n_background=32, n_eval=64, nsamples=512,
    device=device,
)
bee, bar = plot_descriptor_shap(sv, ex, names, SPECIES, EXPLAIN_DIR)
print("E[f] =", round(ev, 4))
display(Image(filename=bee))
display(Image(filename=bar))

In [ ]:
# Method S6 — EdgeSHAPer (per-bond attribution on the molecule); blue = stabilizing, red = destabilizing
EDGE_ROWS = 3                # number of top input rows to explain
EDGE_SAMPLES = 200           # permutation samples per molecule (raise for smoother estimates)

for i in range(min(EDGE_ROWS, len(df))):
    res = edgeshaper(bundle, df.iloc[i], SPECIES, num_samples=EDGE_SAMPLES, device=device)
    png = os.path.join(EXPLAIN_DIR, f"edgeshaper_{SPECIES}_row{i}.png")
    draw_edge_attribution(res, png)
    print(f"row {i}: base={res['base_value']:.3f}  full={res['full_value']:.3f}  -> {png}")
    display(Image(filename=png))

In [ ]:
# Download all explanation figures as a single zip
import shutil
from google.colab import files
shutil.make_archive("mtmm_explanations", "zip", EXPLAIN_DIR)
files.download("mtmm_explanations.zip")

---
### Notes

- In the output, `prob_*` is the positive (unstable) probability and `pred_*` applies the per-species threshold.
- A **structure-only mode** (fingerprint+graph, no SwissADME descriptors) requires a separate
  structure-only model bundle and is not included in this notebook by default.
- A hosted web server / public API is left as future work. This notebook is the accessibility
  deliverable that lets anyone apply the model without local installation.